# 04. Model Training

In this notebook we compare several candidate models for cancer classification. We will keep the model choices modest and interpretable, starting with models that are common for tabular classification tasks.

Candidate models:

- Logistic Regression
- K-Nearest Neighbors
- Support Vector Machine
- Random Forest
- Gradient Boosting

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
data_path = project_root / 'data' / 'data.csv'

df = pd.read_csv(data_path)
df = df.drop(columns=[col for col in df.columns if 'Unnamed:' in str(col)], errors='ignore')
df = df.drop(columns=['id'], errors='ignore')

df['diagnosis'] = df['diagnosis'].astype(str).str.strip()
X = df.drop(columns=['diagnosis'])
y = df['diagnosis'].map({'B': 0, 'M': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000, random_state=42)),
    ]),
    'knn': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5)),
    ]),
    'svm': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', probability=True, random_state=42)),
    ]),
    'random_forest': Pipeline([
        ('scaler', StandardScaler()),
        ('model', RandomForestClassifier(n_estimators=200, random_state=42)),
    ]),
    'gradient_boosting': Pipeline([
        ('scaler', StandardScaler()),
        ('model', GradientBoostingClassifier(random_state=42)),
    ]),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    cv_score = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc').mean()
    print(f'{name}: CV ROC-AUC = {cv_score:.4f}')

## Modeling decisions

This setup keeps the workflow interpretable and reproducible. Logistic regression is a strong baseline for linearly separable problems, while KNN, SVM, and tree-based methods offer different decision boundaries and robustness patterns.

A key principle in this step is that we are comparing models before tuning, not after. This lets us identify which algorithms are promising enough to worth tuning and refining.